## Real Time Currency Conversion

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [11]:
# tool for fetching currency conversion factor

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

# tool for converting currency value from base to target currency using the conversion factor

@tool
def convert(base_currency_value: float, conversion_rate: float) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [12]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [13]:
conversion_rate = get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})
conversion_rate

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772668801,
 'time_last_update_utc': 'Thu, 05 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1772755201,
 'time_next_update_utc': 'Fri, 06 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.2035}

In [14]:
convert.invoke({'base_currency_value':10, 'conversion_rate':conversion_rate['conversion_rate']})

922.0350000000001

In [15]:
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [20]:
messages = [HumanMessage(content="What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR")]
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [21]:
ai_message = llm_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 127, 'total_tokens': 186, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFsiaGC3iljqfbfPlNW4nI2cdXOaK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbbd2-02e5-7043-8b93-4a409f214c0c-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_Zp5Zff8oi4QVtIVLytCTugBV', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 74.29}, 'id': 'call_5RCMW93xEaWrb68L6KgYdSm1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [22]:
messages.append(ai_message)

In [23]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_Zp5Zff8oi4QVtIVLytCTugBV',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 74.29},
  'id': 'call_5RCMW93xEaWrb68L6KgYdSm1',
  'type': 'tool_call'}]

Here, we see that conversion rate is 74.29, not 92.20 as shown above.

Why? LLM read the whole query, and for the first part of the query (conversion rate b/w USD and INR), it called the "get_conversion_factor" tool and got correct tool name and input args. Now, in second part of query (convert 10 USD to INR), LLM realized it needs to suggest "convert" tool, which it did, but for the input args, while it did give base_currency_value as 10 correctly, it couldn't give conversion_rate correctly because till this point we never really made the calculation from the first tool and got the conversion rate. But since LLM needed to give some or the other value for the conversion_rate, it probably looked into its parametric knowledge and found 74.29 as the rate from whenever it had its knowledge cut off.

In such cases, we use InjectedToolArg as the variable type hint, which essentially tells LLM: "hey, don't try to fill this argument, I (the developer), during runtime, will inject this value after running previous tools."

So trying all this again...

In [25]:
# tool for fetching currency conversion factor

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

# tool for converting currency value from base to target currency using the conversion factor

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [27]:
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])
messages = [HumanMessage(content="What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR")]
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)
ai_message.tool_calls


[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_3L2l2LvSa1PtVcESwFfmZLD5',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_LTsnf7MquCn4cZi9GlWnBZlo',
  'type': 'tool_call'}]

Notice there's no conversion_rate in convert tool!

Now, we need to get into this ai_message.tool_calls and calculate conversion_rate. Then use it in 2nd tool. Then prepare messages for LLM.

In [32]:
# pseudo code 
for tool_call in ai_message.tool_calls:
    # execute 1st tool and get conversion_rate
    if tool_call['name'] == 'get_conversion_factor':
        conversion_rate = get_conversion_factor.invoke(tool_call['args'])
        print(conversion_rate['conversion_rate'])
    # execute 2nd tool using conversion_rate from 1st tool
    elif tool_call['name'] == 'convert':
        conversion_result = convert.invoke({'base_currency_value':10, 'conversion_rate':conversion_rate['conversion_rate']})
        print(conversion_result)

92.2035
922.0350000000001


In [ ]:
import json

# preaparing proper messages list for LLM
for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [35]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 122, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DFsxYRaOrORcDZIZelJPYwBhY9OiA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbbe0-2923-7730-aa73-1aab6048a7b5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_3L2l2LvSa1PtVcESwFfmZLD5', 'type': 'tool_call'}, {'name': 'convert', 'args

In [36]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 92.2035. Therefore, converting 10 USD to INR would result in approximately 922.04 INR.'

This is great, but still I made a lot of decisions in between to code the logic out and this wasn't really autonomous and had "making decisions and taking actions" capability, so this is still not an AI Agent.